# RAGAS EVALUATION

In [34]:
import os
import json
import pandas 
import dotenv
from datasets import Dataset
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_correctness,
    context_precision,
    context_recall
)

C:\Users\816032048\AppData\Local\Temp\ipykernel_19816\675308632.py:8: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\816032048\AppData\Local\Temp\ipykernel_19816\675308632.py:8: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import (
C:\Users\816032048\AppData\Local\Temp\ipykernel_19816\675308632.py:8: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\816032048\AppData\Local\

In [35]:
dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_KEY")

print("API key loaded:", os.getenv("OPENAI_API_KEY") is not None)
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_KEY")

with open("../evaluation_results/rag_evaluation_scored_results.json", "r", encoding="utf-8") as f:
    rag_eval_results = json.load(f)

print(f"Loaded {len(rag_eval_results)} evaluation results")
print(rag_eval_results[0].keys())

API key loaded: True
Loaded 75 evaluation results
dict_keys(['question', 'expected_answer', 'generated_answer', 'source_title', 'expected_keywords', 'difficulty', 'retrieved_sources', 'retrieved_contexts', 'source_details', 'source_retrieved_top10', 'keyword_coverage', 'matched_keywords', 'answer_similarity'])


In [36]:
ragas_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

for item in rag_eval_results:
    contexts = item.get("retrieved_contexts", [])

    if isinstance(contexts, str):
        contexts = [contexts]

    ragas_data["question"].append(item["question"])
    ragas_data["answer"].append(item["generated_answer"])
    ragas_data["contexts"].append(contexts)
    ragas_data["ground_truth"].append(item["expected_answer"])

ragas_dataset = Dataset.from_dict(ragas_data)

print(ragas_dataset)
print(ragas_dataset[0])

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 75
})
{'question': 'What problem does the paper address in creating semantic segmentation datasets?', 'answer': 'The paper addresses the problem that creating semantic segmentation datasets is highly challenging due to the extensive manual effort required for accurate pixelwise annotation. This annotation process is laborious, time-consuming, and often requires specialized expertise, especially in domains like medical imaging. Additionally, the scarcity of large, high-quality annotated datasets limits the performance and generalization of deep learning models for semantic segmentation.', 'contexts': ['The fundamental problem of current research is that there is no longer enough data for advanced algorithms or models for special applications. Coupling custom datasets and DL models will be the future theme to many research papers. So many researchers’ outputs consist of not only algorithms or archit

In [37]:
ragas_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    timeout=60,
    max_retries=2
)

ragas_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [38]:
small_dataset = ragas_dataset.select(range(3))

test_scores = evaluate(
    small_dataset,
    metrics=[
        faithfulness,
        answer_correctness,
        context_precision,
        context_recall
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    raise_exceptions=False
)

test_df = test_scores.to_pandas()
display(test_df)

Evaluating: 100%|██████████| 12/12 [00:31<00:00,  2.64s/it]


,user_input,retrieved_contexts,response,reference,faithfulness,answer_correctness,context_precision,context_recall
0,What problem does the paper address in creatin...,[The fundamental problem of current research i...,The paper addresses the problem that creating ...,The paper addresses the high cost and human ef...,0.888889,0.935537,0.887500,1.0
1,What approach is proposed for generating groun...,[Large amounts of high-quality annotated sampl...,An approach proposed for generating ground tru...,The paper proposes creating pixel-accurate sem...,0.777778,0.107163,0.000000,0.0
2,What result did the authors report using game ...,[We begin with experiments on the CamVid datas...,The authors reported that using synthetic game...,Models trained with game data and only one-thi...,1.000000,0.471316,0.866667,1.0


In [39]:
ragas_scores = evaluate(
    ragas_dataset,
    metrics=[
        faithfulness,
        answer_correctness,
        context_precision,
        context_recall
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    raise_exceptions=False
)

ragas_df = ragas_scores.to_pandas()
display(ragas_df)

Evaluating: 100%|██████████| 300/300 [11:23<00:00,  2.28s/it]


,user_input,retrieved_contexts,response,reference,faithfulness,answer_correctness,context_precision,context_recall
0,What problem does the paper address in creatin...,[The fundamental problem of current research i...,The paper addresses the problem that creating ...,The paper addresses the high cost and human ef...,0.888889,0.935537,0.887500,1.0
1,What approach is proposed for generating groun...,[Large amounts of high-quality annotated sampl...,An approach proposed for generating ground tru...,The paper proposes creating pixel-accurate sem...,0.750000,0.107163,0.000000,0.0
2,What result did the authors report using game ...,[We begin with experiments on the CamVid datas...,The authors reported that using synthetic game...,Models trained with game data and only one-thi...,1.000000,0.471316,0.866667,1.0
3,What is the main purpose of the proposed HCI s...,[One of the promising fields in artificial int...,The main purpose of the proposed HCI system is...,The system uses face and hand gesture recognit...,1.000000,0.633736,0.000000,0.0
4,What techniques are used to extract the face a...,"[In the proposed technique, first, the hand ge...",The techniques used to extract the face and ha...,The method combines skin detection and cascade...,1.000000,0.624832,1.000000,1.0
...,...,...,...,...,...,...,...,...
70,What folktale was selected for the animation a...,[People tend to better understand any message ...,"The folktale selected for the animation was ""I...",The story Ijapa ati Atioro was selected becaus...,0.500000,0.593729,0.250000,1.0
71,What software tools were used to develop the a...,[People tend to better understand any message ...,The software tools used to develop the Yoruba ...,The animation was developed using Maya Autodes...,1.000000,0.920486,1.000000,1.0
72,What is the main problem studied in this thesis?,[Chapter 3 reviews literature that is related ...,The main problem studied in this thesis is the...,The thesis studies methods for computing and o...,0.333333,0.103497,0.000000,0.0
73,What two wireless models are used to study cov...,[In this thesis we present new methods for the...,The thesis studies coverage computation using ...,The thesis studies coverage computation using ...,1.000000,0.983064,1.000000,1.0


In [40]:
print("RAGAS Evaluation Summary")
print("=" * 50)

ragas_metric_cols = [
    "faithfulness",
    "answer_correctness",
    "context_precision",
    "context_recall"
]

for col in ragas_metric_cols:
    print(f"{col}: {ragas_df[col].mean():.2%}")

RAGAS Evaluation Summary
faithfulness: 88.28%
answer_correctness: 49.98%
context_precision: 65.64%
context_recall: 66.67%


The high faithfulness score (88.28%) means the generated answers are mostly supported by the retrieved context, so the model is not strongly hallucinating.

The answer correctness score (49.98%) is lower because RAGAS compares the generated answer directly against the reference answer. This suggests that many answers are grounded but may be incomplete, too general, or worded differently from the expected answer.

The context precision score (65.64%) means a fair portion of the retrieved chunks were relevant, but some noisy or less useful chunks were still included.

The context recall score (66.67%) means the retrieved context contained about two-thirds of the information needed to answer the questions fully, but some expected details were missing.

Overall, the RAG system is reliable in terms of grounding, but answer correctness is limited by retrieval completeness and how closely the generated answer matches the  reference answer.

In [41]:
ragas_df.to_csv("../evaluation_results/ragas_evaluation_results.csv", index=False)
print("Saved ../evaluation_results/ragas_evaluation_results.csv")

Saved ../evaluation_results/ragas_evaluation_results.csv


In [42]:
low_correctness = ragas_df.sort_values("answer_correctness").head(10)

display(
    low_correctness[
        [
            "user_input",
            "response",
            "reference",
            "answer_correctness",
            "context_precision",
            "context_recall"
        ]
    ]
)

,user_input,response,reference,answer_correctness,context_precision,context_recall
35,What is the goal of the paper’s benchmark suit...,The goal of the paper’s benchmark suite constr...,The goal is to construct representative GNN mo...,0.067164,0.00,0.0
39,What is the main purpose of the paper?,The main purpose of the paper is to reintroduc...,"The paper gives an overview of cybercrime, exp...",0.079086,0.00,0.0
46,What framework does the paper introduce?,The paper introduces a programming framework d...,It introduces a quad-layered framework for dat...,0.080386,0.00,0.0
12,What is the main focus of this research paper?,The main focus of this research paper is on co...,The paper focuses on mining data in real time ...,0.084348,0.00,0.0
18,What is the purpose of the paper?,The purpose of the paper is to detail the impo...,The paper reflects on lessons from workshops a...,0.097989,0.00,0.0
10,What key model and infrastructure innovations ...,The paper highlights the integration of resili...,The paper highlights Multi-head Latent Attenti...,0.098449,0.50,0.0
36,What framework does the paper improve?,The paper improves the existing computing grid...,The paper improves a static logical framework ...,0.100128,0.25,0.0
53,What result is reported for the best algorithm?,The best algorithm achieves the best results i...,The best algorithm produced many solutions app...,0.100171,0.00,0.0
24,What is the main aim of the paper?,The main aim of the paper is to detail the imp...,The paper compares deep learning and tradition...,0.103088,0.00,0.0
72,What is the main problem studied in this thesis?,The main problem studied in this thesis is the...,The thesis studies methods for computing and o...,0.103497,0.00,0.0


The lowest answer-correctness cases show that most errors were caused by retrieval mismatch rather than hallucination. Several low-scoring rows have context precision and context recall equal to 0.0, meaning the retrieved chunks did not contain the evidence needed to match the reference answer. This occurred mainly for broad or ambiguous questions such as “What is the main purpose of the paper?” or “What framework does the paper introduce?”, where the question depends heavily on retrieving the correct source paper. In these cases, the model generated answers that were faithful to the retrieved context, but the retrieved context came from the wrong or only partially relevant paper. Therefore, the low answer-correctness score is mainly due to incomplete or incorrect context retrieval, not because the LLM was inventing unsupported answers.